# Automatic Differentiation with JAX
`jax` has `numpy`-like syntax but you get gradients for free!

In [ ]:
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)  # match numpy's float64 for a fair, exact comparison below
import jax.numpy as jnp
import time

## `numpy`-like syntax

In [ ]:
def f_np(x):
    return np.sin(x) + x**2

def f_jnp(x):
    return jnp.sin(x) + x**2

print('numpy:', f_np(2.0))
print('jax.numpy:', f_jnp(2.0))

## `jax.grad`: derivatives without doing calculus
`jax.grad(f)` returns a *new function* --- the gradient of `f` --- by tracing every operation `f` performs and applying the chain rule.

In [ ]:
def f(x):
    # some arbitrary function
    return jnp.sin(x) + x**2

df_dx = jax.grad(f)

x0 = 2.0
print('jax.grad(f)(x0)   =', df_dx(x0))
print('analytic derivative =', np.cos(x0) + 2*x0)

Autodiff is exact up to machine precision for *any* `f` unlike finite-difference approximation: 

In [ ]:
def finite_diff(f, x, h):
    return (f(x + h) - f(x - h)) / (2*h)

for h in [1e-2, 1e-4, 1e-6, 1e-8, 1e-12]:
    fd = finite_diff(f, x0, h)
    print(f'h={h:<8.0e}  finite-diff={fd: .10f}   error={abs(fd - df_dx(x0)):.2e}')

print()
print(f'jax.grad = {abs(df_dx(x0) - (np.cos(x0)+2*x0)):.2e}')

## multivariate functions
HMC requires $\nabla\log p(x)$ for a vector $x$. `jax.grad` handles that the same way as above. Let's use the log likelihood from last class exercise of fitting a line to data

In [ ]:
a_true, b_true = 0., 1.75
x_data = jnp.array([1, 1.5, 2, 2.5])
y_data = jnp.array([1.87640523, 2.59001572, 3.4978738, 4.47408932])
yerr = 0.1

def log_likelihood(params, x, y, yerr=0.1):
    a, b = params
    resid = y - (a + b * x)
    return -0.5 * jnp.sum((resid / yerr) ** 2)

grad_log_likelihood = jax.grad(log_likelihood)

params0 = jnp.array([0.5, 1.5])
print('nablda log likelihood(params0) =', grad_log_likelihood(params0, x_data, y_data, yerr))

We can sanity check this analytically for a Gaussian likelihood:

$$\frac{\partial \log p}{\partial a} = \sum_i \frac{y_i - (a+bx_i)}{\varepsilon^2}, \qquad \frac{\partial \log p}{\partial b} = \sum_i \frac{x_i\,(y_i - (a+bx_i))}{\varepsilon^2}$$

In [ ]:
a, b = params0
resid = y_data - (a + b*x_data)
analytic_grad = jnp.array([jnp.sum(resid/yerr**2), jnp.sum(x_data*resid/yerr**2)])

print('jax.grad :', grad_log_likelihood(params0, x_data, y_data, yerr))
print('analytic :', analytic_grad)

## `jax.jit`: just in time compilation
`jax.jit` compiles a function the first time it's called and reuses the compiled version after that, which is much faster

In [ ]:
grad_fast = jax.jit(grad_log_likelihood)

_ = grad_fast(params0, x_data, y_data, yerr)  # first call: compiles

n = 100
t0 = time.time()
for _ in range(n):
    grad_log_likelihood(params0, x_data, y_data, yerr)
t_plain = time.time() - t0
print('normal:', t_plain)

t0 = time.time()
for _ in range(n):
    grad_fast(params0, x_data, y_data, yerr)
t_jit = time.time() - t0

print('jit:', t_jit)